In [0]:
%sql
CREATE OR REPLACE TABLE cms.gold.dim_state
COMMENT 'Prescriber state / area'
AS
SELECT state_code, state_name, area_type
FROM cms.silver.ref_state;

-- A dimension table describes the who, what, and where of each row. The fact table records the how much. dim_state: OH | Ohio | State

In [0]:
%sql
CREATE OR REPLACE TABLE cms.gold.dim_specialty
COMMENT 'Prescriber specialty and group'
AS
SELECT specialty, specialty_group
FROM cms.silver.ref_specialty;
CREATE OR REPLACE TABLE cms.gold.dim_drug
COMMENT 'Drug = brand name + generic name pair. drug_key is a deterministic hash of the pair.'
AS
SELECT
  xxhash64(brand_name, generic_name) AS drug_key,
  brand_name,
  generic_name
FROM (
  SELECT DISTINCT brand_name, generic_name
  FROM cms.silver.partd_prescriber_drug
);
CREATE OR REPLACE TABLE cms.gold.dim_prescriber
COMMENT 'One row per prescriber NPI'
AS
SELECT DISTINCT
  prescriber_npi,
  CASE WHEN is_organization THEN prescriber_last_or_org_name
       ELSE concat_ws(', ', prescriber_last_or_org_name, prescriber_first_name)
  END                         AS prescriber_name,
  is_organization,
  prescriber_city
FROM cms.silver.partd_prescriber_drug;

-- dim_prescriber: 1003000142 | Khalil, Rashid | Toledo, dim_drug: drug_key | Baclofen | Baclofen, dim_specialty: Anesthesiology | Other
-- fact: 2024 | 1003000142 | drug_key | OH | Anesthesiology | 47 | 333.35 (just the numbers and the keys)